# LangChain Multi-Agent Multi-Tool System

LangChain MultiAgent - Updated for the latest LangChain / LangGraph release

This Python application is a basic multi-agent, multi-tool system using LangChain, OpenAI,
Tavily, FAISS, and LangGraph for agent orchestration. It demonstrates specialized agents
for handling technical and math-related queries with advanced input classification and
efficient information retrieval.

Features:
- Technical Agent: Leverages LangChain, OpenAI, Tavily Search, and FAISS for handling technical queries.
- Math Agent: Specialized in math-related questions, equipped with a basic calculator and an equation solver tool.
- LLM Classification: Utilizes a language model (LLM) to classify user queries as 'math' or 'general/technical'.
- Embeddings & FAISS: Employs OpenAI Embeddings and FAISS for efficient information retrieval.
- Tavily Search: Integrated as a search tool to enhance the agent's data access and processing capabilities.

Requirements (latest versions, unpinned):
    pip install -U langchain langchain-openai langchain-community langchain-core langchain-text-splitters \
                langgraph langgraph-prebuilt langchain-tavily python-dotenv faiss-cpu openai tiktoken bs4 gdown

Why unpinned: the previous version of this notebook pinned langchain==1.1.3, which pulled in
langgraph 1.0.10 + langgraph-prebuilt 1.0.13. That specific pinned combination has a real
bug -- langgraph-prebuilt 1.0.13's tool_node.py imports `ExecutionInfo` from `langgraph.runtime`,
which does not exist in langgraph 1.0.10 (it was added later), so importing
`langgraph.prebuilt.create_react_agent` fails with:
    ImportError: cannot import name 'ExecutionInfo' from 'langgraph.runtime'
Installing the current, mutually-compatible releases of langchain/langgraph/langgraph-prebuilt
(verified below: langchain 1.3.15, langgraph 1.2.11, langgraph-prebuilt 1.1.0) avoids this
entirely -- there is no known reason to pin to 1.1.3 here.

Also updated: `langchain_community.tools.tavily_search.TavilySearchResults` is deprecated
(langchain-community itself is being sunset for most integrations) in favor of the standalone
`langchain_tavily` package's `TavilySearch` tool.


In [1]:
# LangChain Multi-Agent System Requirements -- latest, unpinned versions
# Python 3.10+ required

# Core LangChain packages
!pip install -U langchain
!pip install -U langchain-openai
!pip install -U langchain-community
!pip install -U langchain-core
!pip install -U langchain-text-splitters

# LangGraph for agents (required in LangChain 1.x)
!pip install -U langgraph
!pip install -U langgraph-prebuilt

# Tavily search -- standalone package (langchain_community's TavilySearchResults is deprecated)
!pip install -U langchain-tavily

# OpenAI
!pip install -U openai
!pip install -U tiktoken

# Vector Store
!pip install -U faiss-cpu

# Web scraping
!pip install -U beautifulsoup4

# Environment management
!pip install -U python-dotenv

# Google Drive download
!pip install -U gdown


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 970.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 562.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 554.6 kB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.9
    Uninstalling langgraph-1.2.9:
      Successfully uninstalled langgraph-1.2.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━

In [2]:
import os
from dotenv import load_dotenv

# LangChain Core imports
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import create_retriever_tool, tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

# LangChain Community imports
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.chat_message_histories import ChatMessageHistory

# LangChain Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain OpenAI imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# LangGraph imports for agent creation (LangChain 1.x approach)
from langgraph.prebuilt import create_react_agent

# Tavily search -- current standalone integration (replaces the deprecated
# langchain_community.tools.tavily_search.TavilySearchResults)
from langchain_tavily import TavilySearch

import gdown
import os
import warnings
warnings.filterwarnings('ignore')


/tmp/ipykernel_685/3323576048.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [3]:
def download_env_file():
    """Download the .env file from Google Drive."""
    url = 'https://drive.google.com/file/d/1V8-AMoTMsY8nEesspbbuBGxd-O2SnbSh/view?usp=drive_link'
    output_path = '.env'
    gdown.download(url, output_path, quiet=False, fuzzy=True)

In [3]:
def setup_environment():
    """Load environment variables and set up API keys."""
    load_dotenv()
    # NOTE: the value previously hardcoded here for TAVILY_API_KEY looked like a
    # real, live Tavily key -- since it's been pasted into a shared notebook,
    # rotate/revoke it at https://app.tavily.com and put the new one in your
    # .env file instead, the same way OPENAI_API_KEY is loaded.
    # Also: os.environ[...] requires a str -- assigning os.getenv(...) directly
    # crashes with TypeError if the key is missing from .env, so check first.
    for key in ('OPENAI_API_KEY', 'TAVILY_API_KEY'):
        value = os.getenv(key)
        if not value:
            raise RuntimeError(f"{key} not found -- add it to your .env file")
        os.environ[key] = value


In [4]:
def create_first_agent():
    """
    Create the technical agent specialized in handling general/technical queries.
    Uses Tavily Search and FAISS retriever for information retrieval.
    Returns a LangGraph agent.
    """
    # Load documents from web
    loader = WebBaseLoader("https://www.confident-ai.com/docs")
    docs = loader.load()

    # Split documents into chunks
    documents = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    ).split_documents(docs)

    # Create FAISS vector store
    vector = FAISS.from_documents(documents, OpenAIEmbeddings())
    retriever = vector.as_retriever()

    # Create retriever tool
    retriever_tool = create_retriever_tool(
        retriever,
        "confident-ai_Deepeval_search",
        "Search for information about Confident-ai (DeepEval). For any questions about Confident-ai (DeepEval), you must use this tool!",
    )

    # Create Tavily search tool (langchain_tavily.TavilySearch replaces the
    # deprecated langchain_community TavilySearchResults)
    search = TavilySearch(max_results=5)

    # Define tools for the agent
    tools = [search, retriever_tool]

    # Initialize the LLM
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Create the agent using LangGraph's create_react_agent
    # This is the recommended approach in LangChain 1.x
    agent = create_react_agent(
        model=llm,
        tools=tools,
    )

    return agent


In [5]:
@tool
def basic_calculator(query: str) -> str:
    """
    Basic calculator tool that evaluates mathematical expressions.

    Args:
        query: A mathematical expression to evaluate (e.g., "2 + 2", "10 * 5")

    Returns:
        The result of the calculation or an error message.
    """
    try:
        # Using eval with caution - in production, use a safer math parser
        result = eval(query)
        return f"The result is {result}"
    except (SyntaxError, NameError, TypeError, ZeroDivisionError) as e:
        return f"Sorry, I couldn't calculate that due to an error: {e}"


@tool
def equation_solver(query: str) -> str:
    """
    Equation solver tool for solving mathematical equations.

    Args:
        query: The equation to solve (e.g., "3x + 5 = 14")

    Returns:
        The solution or a message indicating the feature status.
    """
    # Basic equation solver (placeholder)
    # In production, implement specific logic for solving equations
    # Consider using sympy for equation solving
    return "Equation solver: This feature is under development."


def create_second_agent():
    """
    Create the math agent specialized in solving math-related questions.
    Equipped with a basic calculator and an equation solver tool.
    Returns a LangGraph agent.
    """
    # Initialize the LLM
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    # Define tools for the math agent
    tools = [basic_calculator, equation_solver]

    # System prompt for math agent
    system_message = "You are specialized in solving math-related questions. Use the available tools to calculate and solve equations. Return the answer to the user's question."

    # Create the agent using LangGraph's create_react_agent
    agent = create_react_agent(
        model=llm,
        tools=tools,
        prompt=system_message,
    )

    return agent

In [6]:
def get_agent(user_input: str) -> str:
    """
    Classify user input to determine which agent should handle it.
    Uses LLM to classify as 'math' or 'general/technical'.

    Args:
        user_input: The user's question or query.

    Returns:
        Classification result: 'math' or 'general/technical'.
    """
    # Initialize the LLM
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    # Create prompt template using LCEL (LangChain Expression Language)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Classify the user input as either "
                   "'math' if it's math-related or 'general/technical' otherwise. "
                   "Respond with ONLY the classification word, nothing else."),
        ("human", "{question}")
    ])

    # Create the chain using LCEL
    chain = prompt | llm | StrOutputParser()

    # Run the chain with the user input
    response = chain.invoke({"question": user_input})

    return response

In [7]:

def run_agent(agent, user_input: str) -> str:
    """
    Run a LangGraph agent with the given input.

    Args:
        agent: The LangGraph agent to run.
        user_input: The user's question or query.

    Returns:
        The agent's response.
    """
    # Create the input message
    messages = [HumanMessage(content=user_input)]

    # Invoke the agent
    result = agent.invoke({"messages": messages})

    # Extract the final response
    if "messages" in result:
        # Get the last AI message
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage):
                return msg.content

    return str(result)


In [16]:
def main():
    """Main function to run the multi-agent system."""
    # Download and load environment variables
    # download_env_file()
    setup_environment()

    # Create agents
    print("Creating technical agent...")
    technical_agent = create_first_agent()

    print("Creating math agent...")
    math_agent = create_second_agent()

    # Example user inputs (uncomment the one you want to test)
    # user_input = "Hello! What's the solution to 3x+5=14?"
    # user_input = "How do confident-ai evaluate complex agentic system?"
    # user_input = "What is agentic ai system?"
    # user_input = "How is LLamaindex?"
    # user_input = "What is 25 * 4 + 100?"
    user_input = " What is 25 * 4 + 100? ?"

    print(f"\nUser Input: {user_input}")
    print("-" * 50)

    # Invoke the agent decider
    response = get_agent(user_input)
    print(f"Classification: {response}")

    if response.strip().lower() == "math":
        print("\n>>> Invoking Math Agent <<<\n")
        result = run_agent(math_agent, user_input)
        print(f"\nResult: {result}")
    else:
        print("\n>>> Invoking Technical Agent <<<\n")
        result = run_agent(technical_agent, user_input)
        print(f"\nResult: {result}")

In [17]:
# Run the main function
if __name__ == '__main__':
    main()

Creating technical agent...
Creating math agent...

User Input:  What is 25 * 4 + 100? ?
--------------------------------------------------
Classification: math

>>> Invoking Math Agent <<<


Result: The result of \(25 \times 4 + 100\) is \(200\).
